# LangChain Redis Kitchen Sink Example

This notebook demonstrates a comprehensive example that combines RedisVectorStore, RedisCache, and RedisChatMessageHistory to create a powerful, efficient, and context-aware chatbot system.

## Setup and Imports

In [ ]:
!pip install -U langchain-redis langchain-openai wikipedia

In [ ]:
import os
from langchain_redis import RedisVectorStore, RedisCache, RedisChatMessageHistory
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain.chains import ConversationalRetrievalChain
import wikipedia

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Redis connection URL
REDIS_URL = "redis://localhost:6379"

## Initialize Components

In [ ]:
# Initialize embeddings
embeddings = OpenAIEmbeddings()

# Initialize RedisVectorStore
vector_store = RedisVectorStore(
    redis_url=REDIS_URL,
    index_name="kitchensink_docs",
    embedding=embeddings
)

# Initialize RedisCache
redis_cache = RedisCache(redis_url=REDIS_URL)
set_llm_cache(redis_cache)

# Initialize ChatOpenAI with caching
llm = ChatOpenAI(temperature=0)

# Initialize RedisChatMessageHistory
message_history = RedisChatMessageHistory("kitchensink_chat", url=REDIS_URL)

## Populate Vector Store with Wikipedia Data

In [ ]:
def fetch_wikipedia_content(titles):
    documents = []
    for title in titles:
        try:
            page = wikipedia.page(title)
            doc = Document(page_content=page.content, metadata={"title": title, "url": page.url})
            documents.append(doc)
        except wikipedia.exceptions.DisambiguationError as e:
            print(f"Disambiguation error for {title}: {e}")
        except wikipedia.exceptions.PageError:
            print(f"Page not found for {title}")
    return documents

# Fetch some Wikipedia articles
titles = ["Artificial Intelligence", "Machine Learning", "Natural Language Processing", "Computer Vision"]
documents = fetch_wikipedia_content(titles)

# Split documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
splits = text_splitter.split_documents(documents)

# Add to vector store
vector_store.add_documents(splits)

print(f"Added {len(splits)} document chunks to the vector store.")

## Create Conversational Retrieval Chain

In [ ]:
retriever = vector_store.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant answering questions based on the provided context. Be concise and accurate."),
    ("human", "{question}")
])

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": prompt}
)

## Interactive Chat Loop

In [ ]:
def get_chat_history(history):
    return [(msg.type, msg.content) for msg in history.messages]

print("Welcome to the AI Assistant! Type 'exit' to end the conversation.")

chat_history = []
while True:
    user_input = input("Human: ")
    if user_input.lower() == 'exit':
        break

    # Add user message to history
    message_history.add_user_message(user_input)

    # Get response from QA chain
    result = qa_chain({"question": user_input, "chat_history": chat_history})
    answer = result['answer']

    # Add AI message to history
    message_history.add_ai_message(answer)

    # Update chat history for context in next iteration
    chat_history = get_chat_history(message_history)

    print(f"AI: {answer}")

    # Print sources (optional)
    sources = set([doc.metadata['title'] for doc in result['source_documents']])
    print(f"Sources: {', '.join(sources)}\n")

print("Thank you for using the AI Assistant!")

## Analysis of the Kitchen Sink Example

This example demonstrates the integration of multiple Redis-based components in LangChain:

1. **RedisVectorStore**: Used to store and retrieve document chunks from Wikipedia articles. It enables efficient similarity search for relevant context.

2. **RedisCache**: Implemented to cache LLM responses, potentially speeding up repeated or similar queries.

3. **RedisChatMessageHistory**: Stores the conversation history, allowing the AI to maintain context across multiple interactions.

The combination of these components creates a powerful, context-aware chatbot system with the following features:

- **Efficient Information Retrieval**: The vector store allows quick access to relevant information from a large dataset.
- **Improved Response Time**: Caching helps in reducing API calls for similar or repeated queries.
- **Contextual Understanding**: The chat history enables the AI to reference previous parts of the conversation.
- **Scalability**: Redis as a backend allows this system to handle large amounts of data and high traffic efficiently.

This kitchen sink example showcases how these Redis-based components can work together seamlessly in a real-world application, demonstrating the power and flexibility of the langchain-redis package.

## Cleanup

In [ ]:
# Clear vector store
vector_store.delete(vector_store.list())

# Clear cache
redis_cache.clear()

# Clear chat history
message_history.clear()

print("Cleanup completed.")